**Intelligent Vision Systems · Lab 6 · Camera Calibration**
Companion to **Lecture 6 — Camera Calibration**
Dr. Muhammad Kazim · Department of Intelligent Systems, University of Lahore

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kazimbalti/intelligent-vision-systems/blob/main/LAB/Lab_6/11-kevin-wood-calibration-app.ipynb) &nbsp; [Lecture 6 slides](https://kazimbalti.github.io/intelligent-vision-systems/lectures/06_Camera_Calibration.html#/watch-and-learn) &nbsp; [All Lab 6 notebooks](https://kazimbalti.github.io/intelligent-vision-systems/labs.html#lab-6-calibration)

# Lab 6K · Camera Calibration Explained: Intrinsics, Extrinsics & Distortion (Kevin Wood)

_Lecture 6, video lab · Kevin Wood._

> 🎬 **Watch first:** [Camera Calibration Explained and SIMPLE Step-by-Step Guide!](https://www.youtube.com/watch?v=Wcnb197g2i0) (Kevin Wood).
> Step-by-step notes for this video: [Tutorial 08 · Calibration explained](https://kazimbalti.github.io/intelligent-vision-systems/tutorials/08_camera_calibration_videos.html#kevin-explained).

<a href="https://www.youtube.com/watch?v=Wcnb197g2i0"><img src="https://img.youtube.com/vi/Wcnb197g2i0/mqdefault.jpg" width="320" alt="Camera Calibration Explained and SIMPLE Step-by-Step Guide!"></a>

> 🎬 **Watch first:** [OpenCV Python Camera Calibration (Intrinsic, Extrinsic, Distortion)](https://www.youtube.com/watch?v=H5qbRTikxI4) (Kevin Wood).
> Step-by-step notes for this video: [Tutorial 08 · OpenCV Python: intrinsics, extrinsics, distortion](https://kazimbalti.github.io/intelligent-vision-systems/tutorials/08_camera_calibration_videos.html#kevin-opencv).

<a href="https://www.youtube.com/watch?v=H5qbRTikxI4"><img src="https://img.youtube.com/vi/H5qbRTikxI4/mqdefault.jpg" width="320" alt="OpenCV Python Camera Calibration (Intrinsic, Extrinsic, Distortion)"></a>

## What the videos do, in plain words

* **World frame → camera frame** is the **extrinsics** (rotation + translation: *where the camera is*).
  **Camera frame → image** is the **intrinsics** (focal length and image centre: *what the camera is like inside*).
  Lens **distortion** (k's = radial, p's = tangential) moves each point a little.
* His **calibration app**: a folder of chessboard pictures → find & draw corners → show **all** pictures in one plot →
  print and save a **JSON** with the reprojection error, camera matrix, distortion, rvecs and tvecs.
* **Remove distortion** with `getOptimalNewCameraMatrix(alpha = 1)` + `undistort`, and **draw a straight line** on the picture
  to *see* the curve disappear.
* **Improve your results:** more pictures, more varied positions/orientations, the **whole** board in view, **no blurry** pictures.

We use the GoPro pictures from Lab 6C and turn his four improvement tips into an automatic **calibration doctor**.

In [ ]:
# Setup (only needed on Google Colab or a fresh environment).
# Uncomment the next line to install the packages this notebook uses:
# !pip install -q opencv-python numpy matplotlib

import os, urllib.request

# Fetch the lab files if they are not next to the notebook.
RAW = "https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/LAB/Lab_6"
for f in ['lab6_checks.py']:
    if not os.path.exists(f):
        os.makedirs(os.path.dirname(f) or ".", exist_ok=True)
        urllib.request.urlretrieve(f"{RAW}/{f}", f)

import lab6_checks as C
C.fetch_set('gopro')
print("data ready")

In [ ]:
import glob, json, os, pickle, time
import numpy as np
import matplotlib.pyplot as plt
import cv2
import lab6_checks as C
np.set_printoptions(precision=4, suppress=True)
print("OpenCV", cv2.__version__)

## 1. The app: find, draw and show all the corners

In [ ]:
nRows, nCols = 8, 6                      # inner corners: count them correctly!
termCriteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)
worldPtsCur = np.zeros((nRows * nCols, 3), np.float32)
worldPtsCur[:, :2] = np.mgrid[0:nRows, 0:nCols].T.reshape(-1, 2)
worldPtsList, imgPtsList, names, drawn = [], [], [], []
imgPathList = sorted(glob.glob("data/calibration_wide/GOPR*.jpg"))
for path in imgPathList:
    imgBGR = cv2.imread(path)
    imgGray = cv2.cvtColor(imgBGR, cv2.COLOR_BGR2GRAY)
    cornersFound, cornersOrg = cv2.findChessboardCorners(imgGray, (nRows, nCols), None)
    if cornersFound:
        worldPtsList.append(worldPtsCur)
        cornersRefined = cv2.cornerSubPix(imgGray, cornersOrg.reshape(-1, 1, 2), (11, 11), (-1, -1), termCriteria)
        imgPtsList.append(cornersRefined); names.append(os.path.basename(path))
        drawn.append(cv2.drawChessboardCorners(imgBGR, (nRows, nCols), cornersRefined, cornersFound))

fig, ax = plt.subplots(5, 7, figsize=(24, 13))
for a in ax.ravel(): a.axis("off")
for a, im, n in zip(ax.ravel(), drawn, names): a.imshow(im[:, :, ::-1]); a.set_title(n, fontsize=9)
plt.show()

## 2. Calibrate and save the JSON (like the app's `calibration.json`)

In [ ]:
h, w = imgGray.shape
repError, camMatrix, distCoeff, rvecs, tvecs = cv2.calibrateCamera(worldPtsList, imgPtsList, (w, h), None, None)
print("Camera Matrix:\n", camMatrix); print("Reproj Error (pixels): {:.4f}".format(repError))
calib = {"repError": repError, "camMatrix": camMatrix.tolist(), "distCoeff": distCoeff.tolist(),
         "rvecs": [r.tolist() for r in rvecs], "tvecs": [t.tolist() for t in tvecs], "image_size": [w, h]}
json.dump(calib, open("calibration.json", "w"), indent=1)
print(list(calib.keys()))

## 3. Intrinsics vs extrinsics, in numbers

Pick one view. Turn its `rvec` into a 3 × 3 rotation matrix and build the full **3 × 4 projection matrix** P = K [R | t]
from the video's equation x = P X. Then project the board's first corner by hand and compare with `projectPoints`
(which also applies distortion).

In [ ]:
j = 0
R, _ = cv2.Rodrigues(rvecs[j])
P = camMatrix @ np.hstack([R, tvecs[j]])
X = np.append(worldPtsCur[0], 1.0)              # homogeneous world point
x = P @ X; u, v = x[:2] / x[2]
uv_cv, _ = cv2.projectPoints(worldPtsCur[:1], rvecs[j], tvecs[j], camMatrix, distCoeff)
print("P (3x4) =\n", P)
print(f"by hand (no distortion): ({u:.1f}, {v:.1f})   projectPoints (with distortion): {uv_cv.ravel().round(1)}   detected: {imgPtsList[j][0, 0].round(1)}")

## 4. Remove the distortion and draw a straight line (alpha = 1, as in the video)

In [ ]:
img = cv2.imread("data/calibration_wide/test_image.jpg")
newCam, roi = cv2.getOptimalNewCameraMatrix(camMatrix, distCoeff, (w, h), 1, (w, h))
und = cv2.undistort(img, camMatrix, distCoeff, None, camMatrix)     # keep K: compare like with like

def straight_line(im, p0, p1):
    im = im.copy(); cv2.line(im, p0, p1, (0, 0, 255), 3); return im

fig, ax = plt.subplots(1, 2, figsize=(20, 7.5))
ax[0].imshow(straight_line(img, (60, 230), (1220, 230))[:, :, ::-1]); ax[0].set_title("distorted: the board edge bends away from the red line")
ax[1].imshow(straight_line(und, (60, 230), (1220, 230))[:, :, ::-1]); ax[1].set_title("undistorted: rows of squares follow straight lines")
for a in ax: a.axis("off")
plt.show()

## 5. The calibration doctor: Kevin's four tips, checked automatically

1. **Enough images?** 2. **Varied positions?** (are all 9 cells of a 3 × 3 grid of the image covered by corners?)
3. **Whole board visible?** (which images failed?) 4. **Blurry images?** Blur score = variance of the Laplacian:
high = sharp, low = blurry. Compute it for every image.

In [ ]:
def blur_score(gray):
    # TODO: return the variance of cv2.Laplacian(gray, cv2.CV_64F)
    return 0.0

scores = [blur_score(cv2.imread(p, 0)) for p in imgPathList]

In [ ]:
C.check_blur_scores(scores, imgPathList)

In [ ]:
def calibration_doctor(paths, found_names, imgpts, scores, size):
    w, h = size
    print(f"1. images: {len(found_names)} usable of {len(paths)}", "OK" if len(found_names) >= 15 else "-> take more (15-30)")
    cells = np.zeros((3, 3), int)
    for c in imgpts:
        for x, y in c.reshape(-1, 2): cells[min(2, int(3 * y / h)), min(2, int(3 * x / w))] += 1
    print("2. corners per image region (3 x 3):\n", cells, "\n   ", "OK" if (cells > 30).all() else "-> put the board in the empty regions")
    failed = [os.path.basename(p) for p in paths if os.path.basename(p) not in found_names]
    print("3. board not found in:", failed or "none", "-> replace these (board cut off / glare)" if failed else "")
    med = np.median(scores); blurry = [os.path.basename(p) for p, s in zip(paths, scores) if s < 0.5 * med]
    print("4. blurry (score < half the median):", blurry or "none")

calibration_doctor(imgPathList, names, imgPtsList, scores, (w, h))

## Your turn

1. Blur three images artificially (`cv2.GaussianBlur(img, (0, 0), 4)`), save them into a copy of the folder and run the
   doctor again. Does it catch them? Does the RMS get worse if you keep them?
2. Load `calibration.json` in a new cell and undistort `test_image.jpg` using only the JSON (no other variables).
3. From section 3: why does the hand projection differ from `projectPoints` by several pixels for this camera, but would
   agree almost perfectly for the dash-cam?